In [36]:
import pandas as pd
df = pd.read_csv("data/hadith.csv")
df

,main_subj,matn_text,url,isnad_text
0,إنما الأعمال بالنيات وإنما لكل امرئ ما نوى,قال سمعت رسول الله صلى الله عليه وسلم يقول إنم...,https://www.islamweb.net/ar/library/content/0/1/,1 حدثنا الحميدي عبد الله بن الزبير قال حدثنا س...
1,أحيانا يأتيني مثل صلصلة الجرس وهو أشده علي فيف...,أن الحارث بن هشام رضي الله عنه سأل رسول الله ص...,https://www.islamweb.net/ar/library/content/0/2/,2 حدثنا عبد الله بن يوسف قال أخبرنا مالك عن هش...
2,اقرأ قال ما أنا بقارئ قال فأخذني فغطني حتى بلغ...,أول ما بدئ به رسول الله صلى الله عليه وسلم من ...,https://www.islamweb.net/ar/library/content/0/3/,4 حدثنا يحيى بن بكير قال حدثنا الليث عن عقيل ع...
3,يعالج من التنزيل شدة وكان مما يحرك شفتيه فقال ...,في قوله تعالى لا تحرك به لسانك لتعجل به قال كا...,https://www.islamweb.net/ar/library/content/0/4/,5 حدثنا موسى بن إسماعيل قال حدثنا أبو عوانة قا...
4,أجود الناس وكان أجود ما يكون في رمضان حين يلقا...,كان رسول الله صلى الله عليه وسلم أجود الناس وك...,https://www.islamweb.net/ar/library/content/0/5/,6 حدثنا عبدان قال أخبرنا عبد الله قال أخبرنا ي...
...,...,...,...,...
6368,ومن أظلم [ ص: 2748 ] ممن ذهب يخلق كخلقي فليخلق...,قال سمعت النبي صلى الله عليه وسلم يقول قال الل...,https://www.islamweb.net/ar/library/content/0/...,7120 حدثنا محمد بن العلاء حدثنا ابن فضيل عن عم...
6369,مثل المؤمن الذي يقرأ القرآن كالأترجة طعمها طيب...,عن النبي صلى الله عليه وسلم قال مثل المؤمن الذ...,https://www.islamweb.net/ar/library/content/0/...,7121 حدثنا هدبة بن خالد حدثنا همام حدثنا قتادة...
6370,تلك الكلمة من الحق يخطفها الجني فيقرقرها في أذ...,سأل أناس النبي صلى الله عليه وسلم عن الكهان فق...,https://www.islamweb.net/ar/library/content/0/...,7122 حدثنا علي حدثنا هشام أخبرنا معمر عن الزهر...
6371,يخرج ناس من قبل المشرق ويقرءون القرآن لا يجاوز...,عن النبي صلى الله عليه وسلم قال يخرج ناس من قب...,https://www.islamweb.net/ar/library/content/0/...,7123 حدثنا أبو النعمان حدثنا مهدي بن ميمون سمع...


In [37]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer
import os
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("hadithMCP")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# ===============================
# Load data & models (ONCE)
# ===============================
df = pd.read_csv("/home/dallash/code/home/hadithMCP/data/hadith.csv")
hadith_embeddings = np.load("/home/dallash/code/home/hadithMCP/data/hadith_embeddings.npy")
print(f"Loaded hadith embeddings: {hadith_embeddings.shape}")

with open("/home/dallash/code/home/hadithMCP/data/bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)

anchor_index = faiss.read_index("/home/dallash/code/home/hadithMCP/data/faiss_anchor.index")
print(f"Anchor index dimension: {anchor_index.d}")

with open("/home/dallash/code/home/hadithMCP/data/anchor_dict.pkl", "rb") as f:
    anchor_dict = pickle.load(f)

with open("/home/dallash/code/home/hadithMCP/data/unique_anchor_texts.pkl", "rb") as f:
    unique_anchor_texts = pickle.load(f)

model = SentenceTransformer("omarelshehy/arabic-english-sts-matryoshka-v2.0")
model.max_seq_length = 512

# Quick dimension check
test_emb = model.encode("test", normalize_embeddings=True)
print(f"Model embedding dimension: {test_emb.shape}")
if test_emb.shape[0] != anchor_index.d:
    raise ValueError(
        f"Dimension mismatch! Model outputs {test_emb.shape[0]}D but "
        f"anchor_index expects {anchor_index.d}D. Rebuild your anchor_index."
    )

from retrieval import hybrid_search_fixed
from utils import preprocess_query

# ===============================
# Helper functions
# ===============================


# ===============================
# Search Function
# ===============================
@mcp.tool()

def search_hadith(query:str, top_k:int)-> list[dict[str,str]]:
    """
    Search a Hadith collection using semantic and keyword-based retrieval.

    This function is designed for AI agents, chatbots, and MCP tools that need
    to retrieve relevant Hadiths from natural language queries. The search uses
    a hybrid retrieval approach combining semantic embeddings and BM25 keyword
    matching to understand the intent and meaning of the user's request rather
    than relying only on exact word matches.

    AI Agent Usage:
        - The agent can provide a question, topic, concept, or description in Arabic.
        - The system converts the query into a semantic representation and retrieves
          Hadiths with similar meanings.
        - Queries do not need to contain exact wording from the Hadith text.

    Example queries:
        "حقوق الوالدين في الإسلام"
        "أحاديث عن الصبر"
        "ما هي الأحاديث التي تتحدث عن الرحمة؟"
        "أحاديث عن فضل طلب العلم"

    Parameters:
        query (str):
            Natural language search query in Arabic. Can be a question,
            topic, concept, or a description of the desired Hadith meaning.

        top_k (int):
            Number of Hadiths to return. Accepted range is 1 to 20.
            Values greater than 20 are automatically limited to 20.

    Returns:
        list[dict]:
            A list of dictionaries containing the retrieved Hadiths.
            Each dictionary has the following fields:

            - "hadith" (str):
                The Hadith text (Matn).
            - "isnad_text" (str):
                The chain of narration (Isnad), if available.
            - "url" (str):
                Reference URL for the Hadith source.

        If the query is empty, returns:
            "<p class='empty'>الرجاء إدخال استعلام بحث</p>"

    Notes:
        - Uses hybrid semantic search (embeddings) and lexical search (BM25).
        - Supports concept-based retrieval, not only exact text matching.
        - Intended for integration with AI agents, MCP servers, RAG systems,
          Islamic assistants, and conversational interfaces.
    """
    if top_k > 20:
        top_k = 20
    elif top_k < 1:
        top_k = 5
    if not query or not str(query).strip():
        return [{"empty":"الرجاء إدخال استعلام بحث"}]

    results_df, debug = hybrid_search_fixed(
        query=query,
        df=df,
        bm25=bm25,
        preprocess_query=preprocess_query,
        model=model,
        hadith_embeddings=hadith_embeddings,
        anchor_index=anchor_index,
        anchor_dict=anchor_dict,
        unique_anchor_texts=unique_anchor_texts,
        top_k=int(top_k)
    )
    results_dict =[]
    for _, row in results_df.iterrows():
        hadith = row["matn_text"]
        isnad_text = row["isnad_text"]
        url = row["url"]
        results_dict.append({"hadith":hadith,
                             "isnad_text": isnad_text,
                             "url"    : url})
    return results_df, debug
    return results_dict



2026-08-04 19:08:28.516950: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loaded hadith embeddings: (6373, 1024)
Anchor index dimension: 1024


[08/04/26 19:08:33] INFO     Use pytorch device_name: cuda:0                             ]8;id=279239;file:///home/dallash/code/myenv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=497976;file:///home/dallash/code/myenv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py#219\219]8;;\

                    INFO     Load pretrained SentenceTransformer:                        ]8;id=680467;file:///home/dallash/code/myenv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=447985;file:///home/dallash/code/myenv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py#227\227]8;;\
                             omarelshehy/arabic-english-sts-matryoshka-v2.0                                        

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Model embedding dimension: (1024,)


In [29]:
x , y =search_hadith(df["matn_text"].loc[5], 5)
print (y["anchor_scores"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[0. 0. 0. ... 0. 0. 0.]


In [13]:
print (y["anchor_scores"].max())

0.999999981593419


In [28]:
df["matn_text"].loc[5]

'أن هرقل أرسل إليه في ركب من قريش وكانوا تجارا بالشأم في المدة التي كان رسول الله صلى الله عليه وسلم ماد فيها أبا سفيان وكفار قريش فأتوه وهم بإيلياء فدعاهم في مجلسه وحوله عظماء الروم ثم دعاهم ودعا بترجمانه فقال أيكم أقرب نسبا بهذا الرجل الذي يزعم أنه نبي فقال أبو سفيان فقلت أنا أقربهم نسبا فقال أدنوه مني وقربوا أصحابه فاجعلوهم عند ظهره ثم قال لترجمانه قل لهم إني سائل هذا عن هذا الرجل فإن كذبني فكذبوه فوالله لولا الحياء من أن يأثروا علي كذبا لكذبت عنه ثم كان أول ما سألني عنه أن قال كيف نسبه فيكم قلت هو فينا ذو نسب قال فهل قال هذا القول منكم أحد قط قبله قلت لا قال فهل كان من آبائه من ملك قلت لا قال فأشراف الناس يتبعونه أم ضعفاؤهم فقلت بل ضعفاؤهم قال أيزيدون أم ينقصون قلت بل يزيدون [ ص: 8 ] قال فهل يرتد أحد منهم سخطة لدينه بعد أن يدخل فيه قلت لا قال فهل كنتم تتهمونه بالكذب قبل أن يقول ما قال قلت لا قال فهل يغدر قلت لا ونحن منه في مدة لا ندري ما هو فاعل فيها قال ولم تمكني كلمة أدخل فيها شيئا غير هذه الكلمة قال فهل قاتلتموه قلت نعم قال فكيف كان قتالكم إياه قلت الحرب بيننا وبينه سجال ينال من

In [25]:
y

{'final_scores': array([0., 0., 0., ..., 0., 0., 0.], shape=(6457,)),
 'anchor_scores': array([0.        , 0.        , 0.        , ..., 0.98734232, 0.98734232,
        0.99999998], shape=(6457,)),
 'semantic_scores': array([0., 0., 0., ..., 0., 0., 0.], shape=(6457,)),
 'bm25_scores': array([0., 0., 0., ..., 0., 0., 0.], shape=(6457,)),
 'union_size': 58}

In [34]:
import numpy as np
k = 10

top_indices = np.argsort(y["anchor_scores"])[-k:][::-1]
print(top_indices)
for i in range(k):
    print(f"{i+1}. {df['matn_text'].iloc[top_indices[i]]}")


[5685 3204 5099 4046 2924 3980 5128 6176 5794 3464]
1. جاء رجل إلى النبي صلى الله عليه وسلم فقال هلكت قال وما شأنك قال وقعت على امرأتي في رمضان قال هل تجد ما تعتق رقبة قال لا قال فهل تستطيع أن تصوم شهرين متتابعين قال لا قال فهل تستطيع أن تطعم ستين مسكينا قال لا أجد فأتي النبي صلى الله عليه وسلم بعرق فيه تمر فقال خذ هذا فتصدق به فقال أعلى أفقر منا ما بين لابتيها أفقر منا ثم قال خذه فأطعمه أهلك
2. كيف فعلتما أتخافان أن تكونا قد حملتما الأرض ما لا تطيق قالا حملناها أمرا هي له مطيقة ما فيها كبير فضل
3. لما كان رسول الله صلى الله عليه وسلم بالطائف قال إنا قافلون غدا إن شاء الله فقال ناس من أصحاب رسول الله صلى الله عليه وسلم لا نبرح أو نفتحها فقال النبي صلى الله عليه وسلم فاغدوا [ ص: 2260 ] على القتال قال فغدوا فقاتلوهم قتالا شديدا وكثر فيهم الجراحات فقال رسول الله صلى الله عليه وسلم إنا قافلون غدا إن شاء الله قال فسكتوا فضحك رسول الله صلى الله عليه وسلم
4. أن النبي صلى الله عليه وسلم قال يوم يقوم الناس لرب العالمين حتى يغيب أحدهم في رشحه إلى أنصاف أذنيه
5. أمر النبي صلى الله عليه وسلم بقتل 

In [ ]:
bm25_scores =[   5 2595 2624 5206 6035   47 2375 2786 3054 5373]

semantic_scores = [   5 2595 3054 4763 2782 3204 5212 5373 3516 3375]
anchor_scores= [5685 3204 5099 4046 2924 3980 5128 6176 5794 3464]



In [35]:
for term in ["الوالدين", "الوالد", "بر الوالدين", "أبويه", "أمه وأبيه"]:
    matches = df[df["matn_text"].str.contains(term, na=False)]
    print(term, "->", len(matches))

الوالدين -> 14
الوالد -> 14
بر الوالدين -> 4
أبويه -> 5
أمه وأبيه -> 0


In [36]:
tokens = preprocess_query("حقوق الوالدين في الاسلام")
print(tokens)
print("الوالدين" in bm25.idf)   # or whatever vocab attribute your BM25 object exposes

['حقوق', 'الوالدين', 'في', 'الاسلام']
True


In [37]:
print(bm25.idf.get("في"))
print(bm25.idf.get("الوالدين"))

0.7123887586794755
6.083556387465009


In [38]:
_, debug = hybrid_search_fixed(
    query="حقوق الوالدين في الاسلام",
    df=df, bm25=bm25, preprocess_query=preprocess_query, model=model,
    hadith_embeddings=hadith_embeddings, anchor_index=anchor_index,
    anchor_dict=anchor_dict, unique_anchor_texts=unique_anchor_texts,
    top_k=10
)
print(np.argsort(debug["bm25_scores"])[::-1][:10])
print(np.argsort(debug["semantic_scores"])[::-1][:10])
print(np.argsort(debug["anchor_scores"])[::-1][:10])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[6345 2352 5581 5751 5798 5217 5752  482 5796 4939]
[3207 6345 5751 5184 5822 4349 4940 5217 2159  482]
[3207 5959 6345  444  482 2159 5184 5822 4940 4349]


In [39]:
for idx in [6345, 5751, 5217, 482]:
    print(idx, "->", df.iloc[idx]["matn_text"][:200])
    print()

6345 -> كان ابن لبعض بنات النبي صلى الله عليه وسلم يقضي فأرسلت إليه أن يأتيها فأرسل إن لله ما أخذ وله ما أعطى وكل إلى أجل مسمى فلتصبر ولتحتسب فأرسلت إليه فأقسمت عليه فقام رسول الله صلى الله عليه وسلم وقمت معه

5751 -> أتي النبي صلى الله عليه وسلم بسكران فأمر بضربه فمنا من يضربه بيده ومنا من يضربه بنعله ومنا من يضربه بثوبه فلما انصرف قال رجل ما له أخزاه الله فقال رسول الله صلى الله عليه وسلم لا تكونوا عون الشيطان عل

5217 -> كان النبي صلى الله عليه وسلم في مسير له فحدا الحادي فقال النبي صلى الله عليه وسلم ارفق يا أنجشة ويحك بالقوارير

482 -> كنا جلوسا عند عمر رضي الله عنه فقال أيكم يحفظ قول رسول الله صلى الله عليه وسلم في الفتنة قلت أنا كما قاله قال إنك عليه أو عليها لجريء قلت فتنة الرجل في أهله وماله وولده وجاره تكفرها الصلاة والصوم والص



In [43]:
for idx in [3207, 5959, 6345,  444,  482, 2159, 5184, 5822, 4940 ,4349]:
    text = df.iloc[idx]["matn_text"]
    print(idx, "الوالدين" in text, "حقوق" in text, "الاسلام" in text)

3207 False False False
5959 False False False
6345 False False False
444 False False False
482 False False False
2159 False False False
5184 False False False
5822 False False False
4940 False False False
4349 False False False


In [45]:
scores = bm25.get_scores(preprocess_query("حقوق الوالدين في الاسلام"))
print("max score:", scores.max())
print("num nonzero:", (scores > 0).sum())
print("num total docs:", len(scores))

# scores of the 14 docs that actually contain "الوالدين"
matches = df[df["matn_text"].str.contains("الوالدين", na=False)].index
print(scores[matches])

max score: 10.380875470522156
num nonzero: 2136
num total docs: 6373


IndexError: index 6429 is out of bounds for axis 0 with size 6373

In [46]:
print("len(df):", len(df))
print("df index min/max:", df.index.min(), df.index.max())
print("bm25 corpus size:", len(bm25.corpus_size) if hasattr(bm25, "corpus_size") else "check your BM25 object's doc count attribute")
print("hadith_embeddings shape:", hadith_embeddings.shape)

len(df): 6457
df index min/max: 0 6456


TypeError: object of type 'int' has no len()

In [51]:
print("bm25 corpus size:", bm25.corpus_size)
print("hadith_embeddings shape:", hadith_embeddings.shape)
print("dataframe shape", len(df))

bm25 corpus size: 6373
hadith_embeddings shape: (6373, 1024)
dataframe shape 6457


In [52]:
# Compare the first N rows' text between df and whatever the embeddings were built from,
# if you don't have that source handy, at least sanity check structurally:
print(df.iloc[6372]["matn_text"][:100])   # last row inside the old embeddings' range
print(df.iloc[6373]["matn_text"][:100])   # first row the old embeddings never saw

كان النبي صلى الله عليه وسلم إذا أتاه السائل وربما قال جاءه السائل أو صاحب الحاجة قال اشفعوا فلتؤجرو
عن النبي صلى الله عليه وسلم قال لا يقل أحدكم اللهم اغفر لي إن شئت ارحمني إن شئت ارزقني إن شئت وليعزم


In [53]:
print(df["matn_text"].duplicated().sum())

84


In [6]:
for i in [5, 3200, 6372, 4000]:
    x, y = search_hadith(df["matn_text"].loc[i], 5)
    print(i, "->", x[:10]["main_subj"])  # does i itself come back on top?

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

5 -> 3204    قال أليس فيكم أو منكم الذي أجاره الله على لسان...
3980    ما بال دعوى الجاهلية قالوا يا رسول الله كسع رج...
5099    كبر الكبر قال يحيى يعني ليلي الكلام الأكبر فتك...
5794    ادعوه فدعوه قال لم لطمت وجهه قال يا رسول الله ...
5128    أتشهد أني رسول الله فنظر إليه فقال أشهد أنك رس...
Name: main_subj, dtype: object


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

6372 -> 5337    كلمتان خفيفتان على اللسان ثقيلتان في الميزان ح...
5587    كلمتان خفيفتان على اللسان [ ص: 2460 ] ثقيلتان ...
6372         كلمتان حبيبتان إلى الرحمن خفيفتان على اللسان
734     إذا قال سمع الله لمن حمده قال اللهم ربنا ولك ا...
564     لا حول ولا قوة إلا بالله وقال هكذا سمعنا نبيكم...
Name: main_subj, dtype: object


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

4000 -> 2840    ثم فتر عني الوحي فترة فبينا أنا أمشي سمعت صوتا...
5162    ثم فتر عني الوحي فبينا أنا أمشي سمعت صوتا من ا...
3999    فبينا أنا أمشي إذ سمعت صوتا من السماء فرفعت رأ...
5207    قوموا إلى سيدكم أو قال خيركم فقعد عند النبي صل...
3       يعالج من التنزيل شدة وكان مما يحرك شفتيه فقال ...
Name: main_subj, dtype: object


In [10]:
bm25.get_scores(preprocess_query(df.loc[4000,"matn_text"]))[4000].max()

np.float64(238.778996084593)

In [3]:
dupe_mask = df["matn_text"].duplicated(keep=False)
print(df[dupe_mask].sort_values("matn_text")[["matn_text"]].head(10))

                                              matn_text
1721  آلى رسول الله صلى الله عليه وسلم من نسائه وكان...
5658  آلى رسول الله صلى الله عليه وسلم من نسائه وكان...
5096  أبلغك أن النبي صلى الله عليه وسلم قال لا حلف ف...
2047  أبلغك أن النبي صلى الله عليه وسلم قال لا حلف ف...
2131  أتيت النبي صلى الله عليه وسلم وهو في المسجد قا...
412   أتيت النبي صلى الله عليه وسلم وهو في المسجد قا...
2230  أعطى رسول الله صلى الله عليه وسلم خيبر اليهود ...
2426  أعطى رسول الله صلى الله عليه وسلم خيبر اليهود ...
2041  أعطى رسول الله صلى الله عليه وسلم خيبر اليهود ...
795   أقبلت راكبا على حمار أتان وأنا يومئذ قد ناهزت ...


In [4]:
df_clean = df.drop_duplicates(subset="matn_text", keep="first").reset_index(drop=True)
print(len(df_clean))  # should be 6373

6373


In [5]:
df_clean.to_csv("/home/dallash/code/home/hadithMCP/data/hadith.csv", index=False)

In [6]:
pd.read_csv("/home/dallash/code/home/hadithMCP/data/hadith.csv").duplicated().sum()

np.int64(0)

In [ ]:
You are evaluating a hadith retrieval system. Below is output from several
test queries, each showing the top 20 results from a  retrieval
methods: SEMANTIC (embedding similarity)

For each result under each method, judge whether the hadith is genuinely
relevant to the query's topic (score 1) or not relevant (score 0). Be strict:
a hadith that merely shares a word with the query but is about a different
subject should score 0.

For each query, report:
- Score out of 20 for SEMANTIC
Then give an overall average per method across all queries, and a short
note on which method seems weakest and why, based on the actual mismatches
you saw (not general assumptions about the method).

Here is the data:
================================================================================


In [ ]:
Here's the relevance scoring for each query and method (1 = genuinely on-topic, 0 = off-topic despite keyword/embedding overlap):

Query	BM25	SEMANTIC	ANCHOR
حقوق الوالدين في الاسلام	4/5	2/5	2/5
فضل الصلاة	0/5	1/5	1/5
أهمية الصدق	1/5	1/5	1/5
بر الوالدين وصلة الرحم	5/5	2/5	1/5
التوبة والاستغفار	0/5	0/5	0/5
أحكام الصيام في رمضان	3/5	2/5	0/5
فضل طلب العلم	0/5	3/5	2/5
التحذير من الكذب والغيبة	2/5	2/5	1/5
الزكاة وحق الفقراء	3/5	3/5	4/5
Total (out of 45)	18	16	12
Average	2.0/5 (40%)	1.8/5 (35.6%)	1.3/5 (26.7%)

Notes on individual calls worth flagging:

BM25's query-4 perfect score and query-1 near-perfect score come from exact phrase overlap ("بر الوالدين", "الكبائر... عقوق الوالدين") — this is BM25 at its best, rewarding literal phrase matches.
I counted [3607] (Ka'b ibn Malik's Tabuk account) as relevant for "lying/backbiting" since that hadith's core content is the reward for truthfulness vs. the excuse-makers' lies — not just a coincidental word match.
[577] ("بين كل أذانين صلاة") was counted relevant to "virtue of prayer" since it's a hadith actively encouraging extra prayer, even though it's not a "reward" statement per se.

Weakest method: ANCHOR, and it's not close (26.7% vs. BM25's 40% and SEMANTIC's 35.6%). The actual mismatches show a consistent pattern: ANCHOR seems to retrieve by narrator/story cluster rather than topic, so once it locks onto the wrong cluster it returns a tight, internally-consistent-but-irrelevant set:

Query 1 (parents' rights) pulled in a Hasan ibn Ali cluster (grandson-related narrations) instead of parent-duty hadiths.
Query 4 (kindness to parents) pulled in the "فداك أبي وأمي" battle-cry cluster — surface overlap on "father and mother" but the actual content is about combat bravery, not filial duty.
Query 6 (Ramadan fasting rulings) pulled in an i'tikaf cluster and generic "gates of paradise open" virtue-hadiths — related to Ramadan broadly, but not to fiqh rulings on fasting specifically.
Query 5 (repentance) totally whiffed (0/5), returning a prayer-etiquette cluster (spitting toward the qibla) that shares no real conceptual ground with tawba/istighfar.

BM25's failures are narrower but more absolute: it scored a flat 0/5 on two queries (فضل الصلاة, فضل طلب العلم) purely because of the polysemous word فضل, which collided with unrelated "surplus of water" and "Aisha's excellence" hadiths. So BM25 is brittle on ambiguous keywords but excellent when the query phrase appears verbatim in the source. SEMANTIC sits in between — it avoided BM25's keyword trap but still failed completely on repentance and only did moderately elsewhere, suggesting the embedding space groups by general "worship/piety" tone rather than the specific act being asked about.
Here's the precision@1 breakdown — whether the single top-ranked result was actually relevant:

Query	BM25 top-1	SEMANTIC top-1	ANCHOR top-1
حقوق الوالدين في الاسلام	✅ [6345] prayer+parents+jihad	❌ [3207] Hasan on the minbar	❌ [5959] Hasan ibn Ali narrative
فضل الصلاة	❌ [5834] surplus water	❌ [532] catching a rak'ah (ruling, not virtue)	✅ [577] "prayer between every adhan"
أهمية الصدق	✅ [5053] truthfulness hadith	❌ [5135] generosity of heart	✅ [5053] truthfulness hadith
بر الوالدين وصلة الرحم	✅ [482] "most beloved deed... parents"	❌ [4145] slave-woman/marriage ruling	❌ [2564] "فداك أبي وأمي" battle-cry
التوبة والاستغفار	❌ [3958] Quranic surah description	❌ [1139] woman weeping at a grave	❌ [1139] same
أحكام الصيام في رمضان	✅ [1791] fasting ruling for tamattu' hajj	❌ [1801] "prayed during Ramadan" (no fasting content)	❌ [1801] same
فضل طلب العلم	❌ [2083] surplus water	✅ [134] "may Allah grant him understanding of religion"	✅ [4091] "best of you learns/teaches Qur'an"
التحذير من الكذب والغيبة	✅ [5053] truthfulness vs. lying	✅ [2376] "signs of the hypocrite: lies when he speaks"	❌ [4753] "avoid shirk and magic" (wrong destructive sin)
الزكاة وحق الفقراء	✅ [2843] "most of paradise's people are poor"	❌ [2111] taking others' money intending repayment	✅ [1310] "every Muslim owes charity"
Precision@1 (out of 9)	6/9 (66.7%)	2/9 (22.2%)	4/9 (44.4%)

What this changes: the picture flips somewhat from the top-5 averages. BM25 actually gets the single best-matching hadith right most of the time (2/3 of queries) — when it works, it works cleanly, and its failures are concentrated in the polysemous فضل queries where it locks onto the wrong sense of the word from result #1 onward. SEMANTIC is the real surprise here: despite a respectable top-5 average, its very first result is relevant only 2 times out of 9 — it tends to surface a thematically adjacent but topically wrong hadith first (grief, marriage rulings, financial disputes) and only recovers relevant hits lower in the list. ANCHOR's precision@1 is better than its top-5 average suggested, because its cluster-anchor sometimes is the right cluster (zakat, knowledge) even when the rest of its top-5 drifts off-topic.

So: BM25 is most trustworthy if you only look at the top hit, but least robust across a full top-5 window on ambiguous terms. SEMANTIC is the opposite — decent spread across 5 results, but don't trust its #1 pick. ANCHOR is weakest overall either way, but its failures are the most explainable (wrong narrator-cluster lock-in) which may make it easiest to fix with better cluster boundaries.

In [50]:
test_queries = [
    "حقوق الوالدين في الاسلام",
    "فضل الصلاة",
    "أهمية الصدق",
    "بر الوالدين وصلة الرحم",
    "التوبة والاستغفار",
    "أحكام الصيام في رمضان",
    "الصبر عند المصائب",
    "حسن الخلق مع الجيران",
    "فضل طلب العلم",
    "التحذير من الكذب والغيبة",
    "الزكاة وحق الفقراء",
    "التوكل على الله",
    "آداب الطعام والشراب",
]

def preview(idx, n=150):
    return df.iloc[idx]["matn_text"][:n].replace("\n", " ")

for q in test_queries:
    _, debug = hybrid_search_fixed(
        query=q, df=df, bm25=bm25, preprocess_query=preprocess_query, model=model,
        hadith_embeddings=hadith_embeddings, anchor_index=anchor_index,
        anchor_dict=anchor_dict, unique_anchor_texts=unique_anchor_texts, top_k=20
    )
    bm25_top = np.argsort(debug["bm25_scores"])[::-1][:5]
    sem_top = np.argsort(debug["semantic_scores"])[::-1][:20]
    anchor_top = np.argsort(debug["anchor_scores"])[::-1][:20]

    print("=" * 80)
    print("QUERY:", q)
 
    print("-- ANCHOR --")
    for i in anchor_top: print(f"  [{i}] {preview(i)}")

    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: حقوق الوالدين في الاسلام
-- ANCHOR --
  [5959] ولقيته بالكوفة وجاء إلى ابن شبرمة فقال أدخلني على عيسى فأعظه فكأن ابن شبرمة خاف عليه فلم يفعل قال حدثنا الحسن قال لما سار الحسن بن علي رضي الله عنهما 
  [3207] سمعت النبي صلى الله عليه وسلم على المنبر والحسن إلى جنبه ينظر إلى الناس مرة وإليه مرة ويقول ابني هذا سيد ولعل الله أن يصلح به بين فئتين من المسلمين
  [6345] أن رجلا سأل النبي صلى الله عليه وسلم أي الأعمال أفضل قال الصلاة لوقتها وبر الوالدين ثم الجهاد في سبيل الله
  [444] لم أعقل أبوي إلا وهما يدينان الدين ولم يمر علينا يوم إلا يأتينا فيه رسول الله صلى الله عليه وسلم طرفي النهار بكرة وعشية ثم بدا لأبي بكر فابتنى مسجدا ب
  [482] سألت النبي صلى الله عليه وسلم أي العمل أحب إلى الله قال الصلاة على وقتها قال ثم أي قال ثم بر الوالدين قال ثم أي قال الجهاد في سبيل الله قال حدثني بهن 
  [2159] أن رسول الله صلى الله عليه وسلم قال المسلم أخو [ ص: 863 ] المسلم لا يظلمه ولا يسلمه ومن كان في حاجة أخيه كان الله في حاجته ومن فرج عن مسلم كربة فرج ال
  [5184] عن النبي صلى الله عليه وسلم قال لا 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: التوكل على الله
-- ANCHOR --
  [1139] مر النبي صلى الله عليه وسلم بامرأة عند قبر وهي تبكي فقال اتقي الله واصبري
  [388] أن النبي صلى الله عليه وسلم رأى نخامة في القبلة فحكها بيده ورئي منه كراهية أو رئي كراهيته لذلك وشدته عليه وقال إن أحدكم إذا قام في صلاته فإنما يناجي ر
  [3122] إلى أبي في منزله فاشترى منه رحلا فقال لعازب ابعث ابنك يحمله معي قال فحملته معه وخرج أبي ينتقد ثمنه فقال له أبي يا أبا بكر حدثني كيف صنعتما حين سريت مع
  [6219] قالوا يا رسول الله إن ها هنا أقواما حديث عهدهم بشرك يأتونا بلحمان لا ندري يذكرون اسم الله عليها أم لا قال اذكروا أنتم اسم الله وكلوا
  [1102] أن النبي صلى الله عليه وسلم رأى نخامة في قبلة المسجد فتغيظ على أهل المسجد وقال إن الله قبل أحدكم فإذا كان في صلاته فلا يبزقن أو قال لا يتنخمن ثم نزل ف
  [4400] أتي رسول الله صلى الله عليه وسلم بطعام ومعه ربيبه عمر بن أبي سلمة فقال سم الله وكل مما يليك
  [3263] عن النبي صلى الله عليه وسلم قال ألا من كان حالفا فلا يحلف إلا بالله فكانت قريش تحلف بآبائها فقال لا تحلفوا بآبائكم
  [2373] أن النبي صلى الله عليه وسل

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: آداب الطعام والشراب
-- ANCHOR --
  [1011] أن رسول الله صلى الله عليه وسلم كان يجمع بين هاتين الصلاتين في السفر يعني المغرب والعشاء
  [1505] أن رسول الله صلى الله عليه وسلم جمع في حجة الوداع المغرب والعشاء بالمزدلفة
  [1009] كان النبي صلى الله عليه وسلم يجمع بين المغرب والعشاء إذا جد به السير
  [2319] كنا مع النبي صلى الله عليه وسلم ثلاثين ومائة فقال النبي صلى الله عليه وسلم هل مع أحد منكم طعام فإذا مع رجل صاع من طعام أو نحوه فعجن ثم جاء رجل مشرك مش
  [4482] عن النبي صلى الله عليه وسلم قال إذا أقيمت الصلاة وحضر العشاء فابدءوا بالعشاء
  [1500] يجمع بين المغرب والعشاء بجمع غير أنه يمر بالشعب الذي أخذه رسول الله صلى الله عليه وسلم فيدخل فينتفض ويتوضأ ولا يصلي حتى يصلي بجمع
  [616] عن النبي صلى الله عليه وسلم أنه قال إذا وضع العشاء وأقيمت الصلاة فابدءوا بالعشاء
  [4481] عن النبي صلى الله عليه وسلم قال إذا وضع العشاء وأقيمت الصلاة فابدءوا بالعشاء
  [4414] قال رسول الله صلى الله عليه وسلم طعام الاثنين كافي الثلاثة وطعام الثلاثة كافي الأربعة
  [618] قال رسول الله صلى الله عليه وسلم إذا 

In [ ]:
You are evaluating a hadith retrieval system. Below is output from several
test queries, each showing the top 20 results from a  retrieval
methods: anchor (embedding similarity)

For each result under each method, judge whether the hadith is genuinely
relevant to the query's topic (score 1) or not relevant (score 0). Be strict:
a hadith that merely shares a word with the query but is about a different
subject should score 0.

For each query, report:
- Score out of 20 for anchor
Then give an overall average per method across all queries, and a short
note on which method seems weakest and why, based on the actual mismatches
you saw (not general assumptions about the method).

Here is the data:
================================================================================


In [ ]:
Here's the SEMANTIC evaluation across all 13 queries (only SEMANTIC results were included in the data, so this covers that method only):

Query	Score /20
آداب الطعام والشراب	6
التوكل على الله	0
فضل الصلاة	4
الزكاة وحق الفقراء	12
حقوق الوالدين في الاسلام	8
التحذير من الكذب والغيبة	5
فضل طلب العلم	8
حسن الخلق مع الجيران	1
الصبر عند المصائب	5
أهمية الصدق	1
أحكام الصيام في رمضان	14
التوبة والاستغفار	1
بر الوالدين وصلة الرحم	9

Overall average: 74/260 = 28.5% (≈5.7/20 per query)

Where it's strong: Ramadan fasting rules (14/20) and zakat/rights of the poor (12/20) — both topics have a lot of surface-lexical redundancy in the corpus (many near-duplicate hadiths with "رمضان" or "زكاة" literally present), which likely helps embedding similarity land on-topic even without deep semantic understanding.

Where it collapses, and why (based on the actual mismatches, not assumption):

أهمية الصدق (truthfulness) — 1/20: Almost every result is pulled from a completely different topic (debt collection, camel/property disputes, oaths on property). This looks like root confusion — the query contains ص-د-ق, and the retriever appears to be matching on tasadduq/sadaqa (charity, "give up a debt") rather than sidq (truthfulness). The one hit ([5053], "truth leads to righteousness...") is the only genuine truthfulness hadith in the whole set.
التوبة والاستغفار (repentance/istighfar) — 1/20: The retrieved set is dominated by hadiths about debt forgiveness and leniency with debtors — "Allah repays for him," "forgiven for easing a debtor's payment." This suggests the embedding is keying on the general concept of "being forgiven" rather than the specific act of repentance/seeking forgiveness (istighfar) as a spiritual practice. Prayer-etiquette hadiths (spitting during salah) also show up repeatedly for no clear thematic reason.
حسن الخلق مع الجيران (good character toward neighbors) — 1/20: Notably, not a single one of the 20 results mentions جار/جيران (neighbor) at all. The retriever substitutes adjacent-but-wrong topics: love/companionship ("a man is with those he loves"), hospitality to travelers, gentleness with animals, and dealings with non-Muslims. It seems to be matching on the general affective register of "good character" while losing the specific object (neighbors) entirely.
التوكل على الله (reliance on Allah) — 0/20: This is the most telling failure. Every result mentions الله somewhere, but the actual content ranges across oaths, prayer etiquette, khaybar battle narratives, and dawah to Mu'adh — none touch the concept of tawakkul. This strongly suggests the semantic embedding is anchoring on the frequency of "الله"/"رسول الله" boilerplate that appears in nearly every hadith isnad/matn, rather than the specific theological concept the query is asking about — a real weakness for a corpus this formulaic.

Diagnosis: SEMANTIC's failure mode isn't random noise — it's systematically pulled toward (a) surface root overlap between semantically distant words sharing letters (sadaqa/sidq), and (b) generic devotional boilerplate ("رسول الله صلى الله عليه وسلم") that saturates nearly every hadith, drowning out the actual topical signal for abstract/values-based queries (tawakkul, sidq, tawbah, neighborliness). It performs far better on queries anchored to concrete nouns/events that recur verbatim across many hadiths (رمضان, الزكاة), where lexical and semantic signal happen to align.

Here's the evaluation of the ANCHOR method's top-20 results across all 12 test queries (التوكل على الله appeared twice with identical results — I've reported it once).

| # | Query | Anchor Score /20 |
|---|---|---|
| 1 | آداب الطعام والشراب | 6 |
| 2 | التوكل على الله | 0 |
| 3 | التحذير من الكذب والغيبة | 2 |
| 4 | فضل طلب العلم | 6 |
| 5 | حسن الخلق مع الجيران | 1 |
| 6 | الصبر عند المصائب | 5 |
| 7 | أحكام الصيام في رمضان | 2 |
| 8 | التوبة والاستغفار | 1 |
| 9 | بر الوالدين وصلة الرحم | 3 |
| 10 | أهمية الصدق | 2 |
| 11 | فضل الصلاة | 3 |
| 12 | حقوق الوالدين في الاسلام | 4 |

**Overall average: 35/240 ≈ 2.9/20 (≈14.6%)**

## Notes on where ANCHOR breaks down

**1. A recurring "debt cluster" acts as filler junk across almost every query.**
Hadiths in the 2113–2128 ID range (all about debt collection, camel-age disputes, an inheritor settling a father's debt, Ja'far/Jabir's debt story, etc.) show up as the bottom 5–10 results in *every single query* — food etiquette, tawakkul, seeking knowledge, Ramadan fasting, honesty, prayer virtue, all of it. That's not noise from a hard borderline call; it's the same ~15 unrelated hadiths being recycled as low-confidence padding regardless of topic, which suggests these items sit in some generic/high-frequency region of the embedding space that the anchor method defaults to when it's not confident.

**2. Prayer-adjacent hadiths (spitting in the qiblah, greeting during prayer, swearing oaths) get pulled for topics with no real connection to prayer mechanics.** This is what tanked *التوكل على الله* to 0/20 — instead of anything about reliance/trust in Allah, the top results were about spitting during prayer, not swearing by one's father, and greeting the Prophet while he prayed. The embedding seems to be keying on generic religious/ritual vocabulary rather than the semantic concept of "tawakkul."

**3. Partial-concept matches score better but still leak.** For *فضل طلب العلم* (6/20) and *الصبر عند المصائب* (5/20), the correct core hadiths were retrieved (e.g., "من يرد الله به خيرا يفقهه في الدين," "الصبر عند الصدمة الأولى"), but they're diluted by roughly half filler — so the method finds the right cluster but doesn't rank tightly within it.

**4. Topic granularity failures**: *حسن الخلق مع الجيران* (neighbors specifically) mostly returned generic "good character" or "hospitality to guests" hadiths rather than anything about neighbors — the anchor method appears to collapse "good character," "hospitality," and "neighbor rights" into one broad semantic neighborhood without distinguishing them.

If you want, I can also score BM25 and SEMANTIC on this same data once you share their outputs, so the three-way comparison and "weakest method" verdict are apples-to-apples.

SyntaxError: unterminated string literal (detected at line 1) (4172946082.py, line 1)

In [12]:
# Pick 3-4 query/hadith pairs you KNOW are genuinely relevant from your eval
# e.g. query "بر الوالدين وصلة الرحم" -> hadith [482] which scored correctly on BM25

query = "بر الوالدين وصلة الرحم"
full_hadith = df.iloc[482]["matn_text"]

# Try encoding the FULL hadith vs. just its first sentence/clause
first_clause = full_hadith.split("قال")[0]  # or split on the first period/clause boundary manually

q_emb = model.encode(query, normalize_embeddings=True)
full_emb = model.encode(full_hadith, normalize_embeddings=True)
clause_emb = model.encode(first_clause, normalize_embeddings=True)

print("similarity to FULL hadith:", q_emb @ full_emb)
print("similarity to FIRST CLAUSE only:", q_emb @ clause_emb)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

similarity to FULL hadith: 0.5274394
similarity to FIRST CLAUSE only: 0.40415025


In [3]:
import re

# Use several KNOWN-relevant query/hadith pairs from your eval results
known_pairs = [
    ("حقوق الوالدين في الاسلام", 6345),
    ("فضل الصلاة", 577),
    ("أهمية الصدق", 5053),
    ("بر الوالدين وصلة الرحم", 482),
    ("أحكام الصيام في رمضان", 1791),
    ("فضل طلب العلم", 134),
    ("فضل طلب العلم", 4091),
    ("التحذير من الكذب والغيبة", 5053),
    ("التحذير من الكذب والغيبة", 2376),
    ("الزكاة وحق الفقراء", 2843),
    ("الزكاة وحق الفقراء", 1310),
]

for query, idx in known_pairs:
    text = df.iloc[idx]["matn_text"]
    word_count = len(text.split())
    q_emb = model.encode(query, normalize_embeddings=True)
    d_emb = model.encode(text, normalize_embeddings=True)
    sim = q_emb @ d_emb
    print(f"len={word_count:4d}  sim={sim:.4f}  idx={idx}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

len=  21  sim=0.5291  idx=6345


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

len=  21  sim=0.2864  idx=2843


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

len=  46  sim=0.6362  idx=1310


In [4]:
import numpy as np

results = []
for query, idx in known_pairs:  # expand this to 10-15 known-relevant pairs from your eval
    text = df.iloc[idx]["matn_text"]
    word_count = len(text.split())
    q_emb = model.encode(query, normalize_embeddings=True)
    d_emb = model.encode(text, normalize_embeddings=True)
    sim = q_emb @ d_emb
    results.append((word_count, sim))

results.sort()
for wc, sim in results:
    print(f"len={wc:4d}  sim={sim:.4f}")

lens = np.array([r[0] for r in results])
sims = np.array([r[1] for r in results])
print("correlation:", np.corrcoef(lens, sims)[0,1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

len=   9  sim=0.6315
len=  20  sim=0.6296
len=  20  sim=0.6600
len=  21  sim=0.2593
len=  21  sim=0.2864
len=  21  sim=0.5291
len=  29  sim=0.4575
len=  36  sim=0.5274
len=  41  sim=0.4794
len=  41  sim=0.5393
len=  46  sim=0.6362
correlation: 0.06716915278990238


In [5]:
import random

for query, idx in known_pairs[:4]:
    q_emb = model.encode(query, normalize_embeddings=True)

    relevant_emb = model.encode(df.iloc[idx]["matn_text"], normalize_embeddings=True)
    relevant_sim = q_emb @ relevant_emb

    random_idx = random.choice(df.index)
    random_emb = model.encode(df.iloc[random_idx]["matn_text"], normalize_embeddings=True)
    random_sim = q_emb @ random_emb

    print(f"query='{query}'")
    print(f"  relevant [{idx}]: {relevant_sim:.4f}")
    print(f"  random   [{random_idx}]: {random_sim:.4f}")
    print(f"  margin: {relevant_sim - random_sim:.4f}\n")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

query='حقوق الوالدين في الاسلام'
  relevant [6345]: 0.5291
  random   [5493]: 0.1358
  margin: 0.3933



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

query='أهمية الصدق'
  relevant [5053]: 0.4794
  random   [3996]: 0.1321
  margin: 0.3473



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

query='بر الوالدين وصلة الرحم'
  relevant [482]: 0.5274
  random   [576]: 0.3418
  margin: 0.1857



In [23]:
for x,y in anchor_dict.items():
    print(x)
    break 

إنما الأعمال بالنيات وإنما لكل امرئ ما نوى


In [31]:
hadith_df = pd.read_csv("/home/dallash/code/home/hadithMCP/data/archive/hadith.csv")

hadith_df = hadith_df.drop_duplicates(subset="matn_text", keep="first").reset_index(drop=True)
print(len(hadith_df))  # should be 6373

6373


In [33]:
print(hadith_df.loc[3207, "main_subj"])   # the Hasan-on-the-minbar hadith that wrongly topped anchor
print(hadith_df.loc[5959, "main_subj"])
print(hadith_df.loc[482, "main_subj"])    # the CORRECT parents hadith

ابني هذا سيد ولعل الله أن يصلح به بين فئتين من المسلمين
ابني هذا سيد ولعل الله أن يصلح به بين فئتين من المسلمين
الصلاة على وقتها قال ثم أي قال ثم بر الوالدين قال ثم أي قال الجهاد في سبيل الله


In [35]:
print(hadith_df['main_subj'].nunique(), "unique subjects out of", len(hadith_df), "rows")
print(hadith_df['main_subj'].value_counts().head(10))

5950 unique subjects out of 6373 rows
main_subj
الولاء لمن أعتق                                                   23
لا ترجعوا بعدي كفارا يضرب بعضكم رقاب بعض                           7
آمركم بأربع وأنهاكم عن أربع                                        6
أنا النبي لا كذب أنا ابن عبد المطلب                                6
الولد للفراش وللعاهر الحجر                                         6
الله أكبر خربت خيبر إنا إذا نزلنا بساحة قوم فساء صباح المنذرين     5
إنما جعل الإمام ليؤتم به                                           5
الخيل لثلاثة لرجل أجر ولرجل ستر وعلى رجل وزر                       4
اللهم أكثر ماله وولده وبارك له فيما أعطيته                         4
لأقضين بينكما بكتاب الله                                           4
Name: count, dtype: int64


In [ ]:
hadith_df = pd.read_csv("/home/dallash/code/home/hadithMCP/data/archive/hadith.csv")

hadith_df = hadith_df.drop_duplicates(subset="matn_text", keep="first").reset_index(drop=True)
print(len(hadith_df))  # should be 6373
import pickle
with open("/home/dallash/code/home/hadithMCP/data/anchor_dict.pkl", "rb") as f:
    anchor_dict = pickle.load(f)
import pandas as pd 
c = 0
for y,x in anchor_dict.items():
    for i in x: 
        if y != hadith_df["main_subj"].loc[i]:
            c+=1
            print(f"the dict embbeding is corrapted at {y} {i}")
print(c)

0


In [27]:
hadith_df["main_subj"].loc[1278]

'أمرت أن أقاتل الناس حتى يقولوا لا إله إلا الله'

In [17]:
hadith_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6457 entries, 0 to 6456
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   _id                 6457 non-null   object 
 1   hadith_number       6457 non-null   int64  
 2   book_id             6457 non-null   object 
 3   narrator_chain_ids  6457 non-null   object 
 4   hadith_title        6083 non-null   object 
 5   ayat_ids            6457 non-null   object 
 6   xref_id             6457 non-null   int64  
 7   xref_url            6088 non-null   object 
 8   islmwy_page         6457 non-null   object 
 9   book_page           6457 non-null   int64  
 10  hadith_status       6457 non-null   object 
 11  book_name           6415 non-null   object 
 12  page_title          6457 non-null   object 
 13  chapter             6457 non-null   object 
 14  chapter_id          6457 non-null   object 
 15  section             511 non-null    object 
 16  sub_se

In [21]:
hadith_df["main_subj"].loc[0]

'إنما الأعمال بالنيات وإنما لكل امرئ ما نوى'

In [58]:
evaluation_queries = {
    "q1":  {"query": "أهمية النية وأثرها في قبول الأعمال", "relevant_ids": [0,1056,5663,5900,5438]},
    "q2":  {"query": "فضل الصدق في القول والعمل وأثره على حياة المؤمن", "relevant_ids": [5191,447,2178,5106,1286]},
    "q3":  {"query": "حقوق الوالدين ووجوب برهما والإحسان إليهما", "relevant_ids": [5650,2368,5825,4997,6429]},
    "q4":  {"query": "أفضل الأعمال وشروط قبول العمل عند الله", "relevant_ids": [4990,4891,1775,1286,4692]},
    "q5":  {"query": "صفات المؤمنين وأثر الإيمان في السلوك", "relevant_ids": [7,46,2481,6224,2917]},
    "q6":  {"query": "الصبر على البلاء واحتساب الأجر", "relevant_ids": [1189,1174,6002,6338,5582]},
    "q7":  {"query": "الإحسان إلى الجار وصلة الرحم", "relevant_ids": [5924,5927,5923,2015,5036]},
    "q8": {"query": "الفرق بين المال الحلال والحرام", "relevant_ids": [1872,3870,1283,2253,6105]},
    "q9": {"query": "خطر الحسد وطرق الوقاية منه", "relevant_ids": [6424,6067,5031,1285,69]},
    "q10": {"query": "فضل الصدقة وأثرها في تطهير المال", "relevant_ids": [4418,1301,1280,2067]},
    "q11": {"query": "تحريم الكذب وفضل الصدق", "relevant_ids": [5106,3845,2393,4379,1180]},
    "q12": {"query": "فضل نشر العلم وتعليمه", "relevant_ids": [94,6001,4315,79,75]},
    "q13": {"query": "فضل الصلاة في وقتها وأثرها على المسلم", "relevant_ids": [53,732,626,6429,460]},
    "q14": {"query": "رعاية اليتيم وفضل الإحسان إليه", "relevant_ids": [1857,2017,5003,4376,5023]},
    "q15": {"query": "فضل التواضع والتحذير من الكبر", "relevant_ids": [6421,5325,603,2313,65]},
    "q16": {"query": "الرفق والرحمة في معاملة الناس", "relevant_ids": [6282,5031,3242,2416,6350]},
    "q17": {"query": " الاستغفار والتوبة ", "relevant_ids": [2974,1212,1050,6350,149]},
    "q18": {"query": "التوكل على الله مع الأخذ بالأسباب", "relevant_ids": [4692,73,46,22,2414]},
    "q19": {"query": "حقوق العمال والخدم والعدل في معاملتهم", "relevant_ids": [417,2249,2223,2027,2026]},
    "q20": {"query": " مساعدة الفقراء والإحسان إليهم", "relevant_ids": [2471,1347,2139,2702,5025]},

}
test_queries =[]
for x in evaluation_queries.values():
    test_queries.append(x["query"])

In [47]:
import numpy as np

anchor_vals = []
semantic_vals = []

for q in test_queries:
    _, debug = hybrid_search_fixed(
        query=q, df=df, bm25=bm25, preprocess_query=preprocess_query, model=model,
        hadith_embeddings=hadith_embeddings, anchor_index=anchor_index,
        anchor_dict=anchor_dict, unique_anchor_texts=unique_anchor_texts, top_k=10
    )
    anchor_vals.extend(debug["anchor_scores"])
    semantic_vals.extend(debug["semantic_scores"])

print("correlation between anchor and semantic scores:", np.corrcoef(anchor_vals, semantic_vals)[0,1])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

correlation between anchor and semantic scores: 0.609217980541914


In [52]:
import re

# Common recurring formulaic phrases in hadith collections.
# Order matters: longer/more specific phrases first, so a shorter
# phrase doesn't partially eat a longer one and leave fragments behind.
BOILERPLATE_PHRASES = [
    "صلى الله عليه وسلم",
    "رضي الله عنه",
    "رضي الله عنها",
    "رضي الله عنهما",
    "رضي الله عنهم",
    "عليه السلام",
    "عز وجل",
    "تبارك وتعالى",
]

def strip_boilerplate(text):
    if not isinstance(text, str):
        return text
    cleaned = text
    for phrase in BOILERPLATE_PHRASES:
        cleaned = cleaned.replace(phrase, " ")
    # collapse any double spaces left behind by the removals
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned

df["clean_text"] = df["matn_text"].apply(strip_boilerplate)

# sanity check: look at a few before/after pairs before trusting it
for i in [5, 482, 5053]:
    print("BEFORE:", df.iloc[i]["matn_text"][:150])
    print("AFTER :", df.iloc[i]["clean_text"][:150])
    print()

BEFORE: أن هرقل أرسل إليه في ركب من قريش وكانوا تجارا بالشأم في المدة التي كان رسول الله صلى الله عليه وسلم ماد فيها أبا سفيان وكفار قريش فأتوه وهم بإيلياء فد
AFTER : أن هرقل أرسل إليه في ركب من قريش وكانوا تجارا بالشأم في المدة التي كان رسول الله ماد فيها أبا سفيان وكفار قريش فأتوه وهم بإيلياء فدعاهم في مجلسه وحوله

BEFORE: سألت النبي صلى الله عليه وسلم أي العمل أحب إلى الله قال الصلاة على وقتها قال ثم أي قال ثم بر الوالدين قال ثم أي قال الجهاد في سبيل الله قال حدثني بهن 
AFTER : سألت النبي أي العمل أحب إلى الله قال الصلاة على وقتها قال ثم أي قال ثم بر الوالدين قال ثم أي قال الجهاد في سبيل الله قال حدثني بهن ولو استزدته لزادني

BEFORE: عن النبي صلى الله عليه وسلم قال إن الصدق يهدي إلى البر وإن البر يهدي إلى الجنة وإن الرجل ليصدق حتى يكون صديقا وإن الكذب يهدي إلى الفجور وإن الفجور يهد
AFTER : عن النبي قال إن الصدق يهدي إلى البر وإن البر يهدي إلى الجنة وإن الرجل ليصدق حتى يكون صديقا وإن الكذب يهدي إلى الفجور وإن الفجور يهدي إلى النار وإن الر



In [ ]:
clean_embeddings = model.encode(
    df["clean_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32,
)

Batches:   0%|          | 0/200 [00:00<?, ?it/s]

In [55]:
type(clean_embeddings)

numpy.ndarray

In [56]:
DATA_DIR =r"/home/dallash/code/home/hadithMCP/data"
np.save(f"{DATA_DIR}/clean_embeddings.npy", clean_embeddings)

In [60]:
"""test_queries = [
    "حقوق الوالدين في الاسلام",
    "فضل الصلاة",
    "أهمية الصدق",
    "بر الوالدين وصلة الرحم",
    "التوبة والاستغفار",
    "أحكام الصيام في رمضان",
    "الصبر عند المصائب",
    "حسن الخلق مع الجيران",
    "فضل طلب العلم",
    "التحذير من الكذب والغيبة",
    "الزكاة وحق الفقراء",
    "التوكل على الله",
    "آداب الطعام والشراب",
]"""

def preview(idx, n=150):
    return df.iloc[idx]["matn_text"][:n].replace("\n", " ")

for q in test_queries:
    _, debug = hybrid_search_fixed(
        query=q, df=df, bm25=bm25, preprocess_query=preprocess_query, model=model,
        hadith_embeddings=hadith_embeddings, anchor_index=anchor_index,
        anchor_dict=anchor_dict, unique_anchor_texts=unique_anchor_texts, top_k=20
    )
    bm25_top = np.argsort(debug["bm25_scores"])[::-1][:5]
    sem_top = np.argsort(debug["semantic_scores"])[::-1][:20]
    anchor_top = np.argsort(debug["anchor_scores"])[::-1][:20]

    print("=" * 80)
    print("QUERY:", q)
 
    print("-- sematic --")
    for i in sem_top: print(f"  [{i}] {preview(i)}")

    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: أهمية النية وأثرها في قبول الأعمال
-- sematic --
  [5135] قال رسول الله صلى الله عليه وسلم ويقولون الكرم إنما الكرم قلب المؤمن
  [2007] قال النبي صلى الله عليه وسلم الخازن الأمين الذي يؤدي ما أمر به طيبة نفسه أحد المتصدقين
  [4843] أن النبي صلى الله عليه وسلم كان يحتجر حصيرا بالليل فيصلي عليه ويبسطه بالنهار فيجلس عليه فجعل الناس يثوبون إلى النبي صلى الله عليه وسلم فيصلون بصلاته ح
  [4379] أن رسول الله صلى الله عليه وسلم قال خير الصدقة ما كان عن ظهر غنى وابدأ بمن تعول
  [1294] عن النبي صلى الله عليه وسلم قال خير الصدقة ما كان عن ظهر غنى وابدأ بمن تعول
  [0] قال سمعت رسول الله صلى الله عليه وسلم يقول إنما الأعمال بالنيات وإنما لكل امرئ ما نوى فمن كانت هجرته إلى دنيا يصيبها أو إلى امرأة ينكحها فهجرته إلى ما
  [3304] سمعت النبي صلى الله عليه وسلم يقول الأعمال بالنية فمن كانت هجرته إلى دنيا يصيبها أو امرأة يتزوجها فهجرته إلى ما هاجر إليه ومن كانت هجرته إلى الله ورسو
  [50] أن رسول الله صلى الله عليه وسلم قال الأعمال بالنية ولكل امرئ ما نوى فمن كانت هجرته إلى الله ورسوله فهجرته إلى ال

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: حقوق العمال والخدم والعدل في معاملتهم
-- sematic --
  [2007] قال النبي صلى الله عليه وسلم الخازن الأمين الذي يؤدي ما أمر به طيبة نفسه أحد المتصدقين
  [2233] أن رسول الله صلى الله عليه وسلم قال من أعتق شركا له في عبد فكان له مال يبلغ ثمن العبد قوم العبد عليه قيمة عدل فأعطى شركاءه حصصهم وعتق عليه العبد وإلا 
  [2026] كان النبي صلى الله عليه وسلم يحتجم ولم يكن يظلم أحدا أجره
  [2260] عن النبي صلى الله عليه وسلم قال المملوك الذي يحسن عبادة ربه ويؤدي إلى سيده الذي له عليه من الحق والنصيحة والطاعة له أجران
  [2655] عن النبي صلى الله عليه وسلم قال ثلاثة يؤتون أجرهم مرتين الرجل تكون له الأمة فيعلمها فيحسن تعليمها ويؤدبها فيحسن أدبها ثم يعتقها فيتزوجها فله أجران ومؤ
  [2017] عن النبي صلى الله عليه وسلم قال مثل المسلمين واليهود والنصارى كمثل رجل استأجر قوما يعملون له عملا يوما إلى الليل على أجر معلوم فعملوا له إلى نصف [ ص: 
  [2015] أن رسول الله صلى الله عليه وسلم قال إنما مثلكم واليهود والنصارى كرجل استعمل عمالا فقال من يعمل لي إلى نصف النهار على قيراط قيراط فعملت اليهود على قيرا
  [91] 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY:  مساعدة الفقراء والإحسان إليهم
-- sematic --
  [5824] أن رسول الله صلى الله عليه وسلم قال المسلم أخو المسلم لا يظلمه ولا يسلمه ومن كان في حاجة أخيه كان الله في حاجته
  [2450] أن عمر رضي الله عنه وجد مالا بخيبر فأتى النبي صلى الله عليه وسلم فأخبره قال إن شئت تصدقت بها فتصدق بها في الفقراء والمساكين وذي القربى والضيف
  [2125] قال رسول الله صلى الله عليه وسلم أو قال سمعت رسول الله صلى الله عليه وسلم يقول من أدرك ماله بعينه عند رجل أو إنسان قد أفلس فهو أحق به من غيره
  [4374] عن النبي صلى الله عليه وسلم قال إذا أنفق المسلم نفقة على أهله وهو يحتسبها كانت له صدقة
  [5183] أن رجلا سأل النبي صلى الله عليه وسلم أي الإسلام خير قال تطعم الطعام وتقرأ السلام على من عرفت وعلى من لم تعرف
  [26] أن رجلا سأل رسول الله صلى الله عليه وسلم أي الإسلام خير قال تطعم الطعام وتقرأ السلام على من عرفت ومن لم تعرف
  [10] أن رجلا سأل النبي صلى الله عليه وسلم أي الإسلام خير قال تطعم الطعام وتقرأ السلام على من عرفت ومن لم تعرف
  [1553] أن رسول الله صلى الله عليه وسلم قال اللهم ارحم المحلقين قالوا والمقصرين يا

In [61]:
sticky_ids = [5135]  # add the second one once you find its index

for idx in sticky_ids:
    text = df.iloc[idx]["matn_text"]
    print(f"idx={idx}  len={len(text.split())} words")
    print(text)
    print()

# Check: how close is this hadith's embedding to the AVERAGE embedding of the whole corpus?
# A hub sitting near the centroid would score unusually high against almost anything.
centroid = hadith_embeddings.mean(axis=0)
centroid = centroid / np.linalg.norm(centroid)

for idx in sticky_ids:
    sim_to_centroid = hadith_embeddings[idx] @ centroid
    print(f"idx={idx}  similarity to corpus centroid: {sim_to_centroid:.4f}")

# Compare against a few random hadiths for reference
random_sample = np.random.choice(len(df), 20, replace=False)
random_sims = [hadith_embeddings[i] @ centroid for i in random_sample]
print("average random similarity to centroid:", np.mean(random_sims))

idx=5135  len=13 words
قال رسول الله صلى الله عليه وسلم ويقولون الكرم إنما الكرم قلب المؤمن

idx=5135  similarity to corpus centroid: 0.5077
average random similarity to centroid: 0.55266714


In [62]:
from collections import Counter

top1_counter = Counter()

for q in test_queries:  # your 20-query set
    q_emb = model.encode(q, normalize_embeddings=True)
    sims = hadith_embeddings @ q_emb
    top1_idx = sims.argmax()
    top1_counter[top1_idx] += 1

print(top1_counter.most_common(10))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[(np.int64(2258), 3), (np.int64(5135), 2), (np.int64(2262), 2), (np.int64(1139), 2), (np.int64(2992), 1), (np.int64(5822), 1), (np.int64(4268), 1), (np.int64(4692), 1), (np.int64(5282), 1), (np.int64(5053), 1)]


In [63]:
sticky_ids = [idx for idx, count in top1_counter.most_common(10) if count > 1]
sticky_lengths = [len(df.iloc[i]["matn_text"].split()) for i in sticky_ids]
all_lengths = df["matn_text"].str.split().str.len()

print("sticky hadith lengths:", sticky_lengths)
print("corpus median length:", all_lengths.median())
print("corpus 10th percentile length:", all_lengths.quantile(0.1))

sticky hadith lengths: [14, 13, 30, 15]
corpus median length: 31.0
corpus 10th percentile length: 13.0


In [64]:
print(df.iloc[2258]["matn_text"])
print(df.iloc[2262]["matn_text"])  # the outlier, 30 words, check why IT is sticky too

قال النبي صلى الله عليه وسلم نعم ما لأحدهم يحسن عبادة ربه وينصح لسيده
قال النبي صلى الله عليه وسلم من أعتق نصيبا له من العبد فكان له من المال ما يبلغ قيمته يقوم عليه قيمة عدل وأعتق من ماله وإلا فقد عتق منه


In [65]:
# CPU-friendly, retrieval-tuned, multilingual
from sentence_transformers import SentenceTransformer
e5_model = SentenceTransformer("intfloat/multilingual-e5-small")

# e5 models require query:/passage: prefixes — this alone may help vs your current model's total lack of asymmetry awareness
q_emb = e5_model.encode("query: التوكل على الله", normalize_embeddings=True)
d_embs = e5_model.encode(["passage: " + t for t in df["matn_text"].tolist()[:200]], normalize_embeddings=True)  # small subset first

[08/05/26 02:05:55] INFO     Use pytorch device_name: cuda:0                             ]8;id=188516;file:///home/dallash/code/myenv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=958939;file:///home/dallash/code/myenv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py#219\219]8;;\

                    INFO     Load pretrained SentenceTransformer:                        ]8;id=641476;file:///home/dallash/code/myenv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py\SentenceTransformer.py]8;;\:]8;id=179548;file:///home/dallash/code/myenv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py#227\227]8;;\
                             intfloat/multilingual-e5-small                                                        

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [67]:
sims = d_embs @ q_emb
top5 = np.argsort(sims)[::-1][:5]

for i in top5:
    print(sims[i], df.iloc[i]["matn_text"][:150])

0.8771607 توضأ النبي صلى الله عليه وسلم مرة مرة
0.86843497 أن النبي صلى الله عليه وسلم توضأ مرتين مرتين
0.8643828 حفظت من رسول الله صلى الله عليه وسلم وعاءين
0.86083555 عن النبي صلى الله عليه وسلم أنه مسح على الخفين
0.85963565 بينما نحن جلوس مع النبي صلى الله عليه وسلم في المسجد دخل رجل على جمل فأناخه في المسجد ثم عقله ثم قال لهم أيكم محمد والنبي صلى الله عليه وسلم متكئ بين 
